In [ ]:
import json
import requests
import xml.etree.ElementTree as ET
import torch
from transformers import AutoTokenizer, AutoModel
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load BioASQ test questions
def load_bioasq_test_questions(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data['questions']

# 2. Fetch PubMed documents (title + abstract)
def fetch_pubmed_docs(query, max_results=50, api_key="44c6e8c525237134d87515d972ff3ad85608"):
    # Search for IDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": max_results
    }
    if api_key:
        search_params["api_key"] = api_key
    resp = requests.get(search_url, params=search_params)
    resp.raise_for_status()
    id_list = resp.json()["esearchresult"]["idlist"]
    if not id_list:
        return []
    # Fetch details
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(id_list),
        "retmode": "xml"
    }
    resp = requests.get(fetch_url, params=fetch_params)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    docs = []
    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID")
        title = article.findtext(".//ArticleTitle") or ""
        abstract = " ".join([abst.text or "" for abst in article.findall(".//AbstractText")])
        docs.append({
            "pmid": pmid,
            "title": title,
            "documentAbstract": abstract
        })
    return docs

# 3. Get BERT embedding for a text
def get_embedding(text, model, tokenizer, device, max_length=256):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb

# 4. Rank documents by cosine similarity to question
def rank_documents(question, docs, model, tokenizer, device):
    q_emb = get_embedding(question, model, tokenizer, device)
    doc_embs = [get_embedding(doc["title"] + " " + doc["documentAbstract"], model, tokenizer, device) for doc in docs]
    sims = cosine_similarity([q_emb], doc_embs)[0]
    ranked = sorted(zip(docs, sims), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked], [score for doc, score in ranked]

# 5. Extract and rank snippets
def extract_and_rank_snippets(question, docs, model, tokenizer, device, max_snippets=10):
    q_emb = get_embedding(question, model, tokenizer, device)
    snippets = []
    for doc in docs:
        text = doc["title"] + " " + doc["documentAbstract"]
        sentences = sent_tokenize(text)
        offset = 0
        for sent in sentences:
            sent_emb = get_embedding(sent, model, tokenizer, device)
            sim = cosine_similarity([q_emb], [sent_emb])[0][0]
            snippets.append({
                "document": f"http://www.ncbi.nlm.nih.gov/pubmed/{doc['pmid']}",
                "text": sent,
                "offsetInBeginSection": offset,
                "offsetInEndSection": offset + len(sent),
                "beginSection": "abstract",
                "endSection": "abstract",
                "score": sim
            })
            offset += len(sent) + 1
    # Rank and return top N
    snippets = sorted(snippets, key=lambda x: x["score"], reverse=True)[:max_snippets]
    for s in snippets:
        del s["score"]  # Remove score for BioASQ format
    return snippets

# 6. Main pipeline
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

bioasq_13b_test_questions = load_bioasq_test_questions('../data/BioASQ-task13bPhaseA-testset4.txt')
results = []

for question in bioasq_13b_test_questions:
    print(f"Processing question: {question['id']}")
    docs = fetch_pubmed_docs(question['body'], max_results=50)
    if not docs:
        print("No docs found.")
        continue
    ranked_docs, _ = rank_documents(question['body'], docs, model, tokenizer, device)
    top_docs = ranked_docs[:10]
    ranked_snippets = extract_and_rank_snippets(question['body'], top_docs, model, tokenizer, device, max_snippets=10)
    question_result = {
        'id': question['id'],
        'documents': [f"http://www.ncbi.nlm.nih.gov/pubmed/{doc['pmid']}" for doc in top_docs],
        'snippets': ranked_snippets
    }
    results.append(question_result)
    print(f"Found {len(top_docs)} documents and {len(ranked_snippets)} snippets")

with open('BioASQ-task13b-phaseA-testset4-neural-results2.json', 'w') as f:
    json.dump({'questions': results}, f, indent=2)

Processing question: 67e6cf2618b1e36f2e0000d0
Found 3 documents and 10 snippets
Processing question: 680d5e47353a4a2e6b000005


/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Found 10 documents and 10 snippets
Processing question: 680f4a68353a4a2e6b000007


/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Found 10 documents and 10 snippets
Processing question: 680a083218b1e36f2e00014d


...

Found 10 documents and 10 snippets
Processing question: 67e090be18b1e36f2e00006c


/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Found 10 documents and 10 snippets
Processing question: 67d7fded18b1e36f2e000042


/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/khasrurrahman/Library/CloudStorage/OneDrive-DaffodilInternationalUniversity/TU Wine/2025S/Advanced Information Retrieval/Group Assignment/bioasq-13b/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Found 10 documents and 10 snippets
